### Graph Kernels (GraKel)

expand patient graph representations for sick/ not sick prediction

to Do (inspired by Johannes testing.ipynb)
- parameter grid to optimize kernels and classfiers
- include more kernels and classifiers

In [1]:
NEO4J_URI = "neo4j://83.229.84.12"
NEO4J_USERNAME = "tumaiReadonly"
NEO4J_PASSWORD = "MAKEATHON2024"
NEO4J_DB = "graph2.db"

OUTPUT_DIR = "data"

In [18]:
# Function to extract graph data from Neo4j
from neo4j import GraphDatabase

# Function to connect to Neo4j
def connect_to_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    return driver

def extract_graphs_from_neo4j(driver):
    query = """
    MATCH (bs:Biological_sample)
    //WITH bs LIMIT 10
    MATCH (bs)-[r_prot:HAS_PROTEIN]->(prot:Protein)
    MATCH (bs)-[r_pheno:HAS_PHENOTYPE]->(pheno:Phenotype)
    MATCH (bs)-[r_gene:HAS_DAMAGE]->(gene:Gene)
        OPTIONAL MATCH (bs)-[r_disease:HAS_DISEASE]->(d:Disease)
        //WHERE NOT (bs)-[:HAS_DISEASE]->(d:Disease {name:"control"})
        RETURN DISTINCT bs.subjectid AS subjectid, prot.id AS protein_id, r_prot.score AS protein_score, id(bs) AS node_bs, id(prot) AS node_prot, pheno.id AS phenotype, r_pheno.score AS pheno_score,
        gene.id AS gene, r_gene.score AS gene_score, id(pheno) AS node_pheno, id(gene) AS node_gene, 
        type(r_prot) AS rel_prot, type(r_pheno) AS rel_pheno, type(r_gene) AS rel_gene,
        CASE WHEN d IS NULL THEN 0 ELSE 1 END AS isSick"""
 
    graphs_isSick = {}
    with driver.session(database=NEO4J_DB) as session:
        result = session.run(query)
        
        # Create a dictionary to store nodes and their labels
        for record in result:

            node_bs = record["node_bs"]
            node_prot = record["node_prot"]
            node_gene = record["node_gene"]
            node_pheno = record["node_pheno"]

            rel_prot = record["rel_prot"]
            rel_pheno = record["rel_pheno"]
            rel_gene = record["rel_gene"]

            subjectid = record["subjectid"]
            protein_id = record["protein_id"]
            phenotype = record["phenotype"]
            gene_id = record["gene"]
            
            protein_score = record["protein_score"]
            if protein_score is None:
                protein_score = 1
            else: protein_score = protein_score

            gene_score = record["gene_score"]
            if gene_score is None:
                gene_score = 1
            else: gene_score = gene_score

            pheno_score = record["pheno_score"]
            if pheno_score is None:
                pheno_score = 1 #for now, we don't have a score for phenotypes
            else: pheno_score = pheno_score

            isSick = record["isSick"]
            
            # If we haven't seen this subjectid before, initialize a new graph entry
            if subjectid not in graphs_isSick:
                graphs_isSick[subjectid] = {
                    'isSick': isSick,
                    'edges': set(),  # Change to a set to track unique edges
                    'nodes': {},
                    'edge_labels': {}
                }
            
            # Add the phenotype as a node and the relationship (bs -> p) as an edge
            graphs_isSick[subjectid]['nodes'][node_bs] = 'subjectid'  # just refers to the Biological_sample node as 'subjectid'
            graphs_isSick[subjectid]['nodes'][node_prot] = protein_id  # protein_id as a node label
            graphs_isSick[subjectid]['nodes'][node_gene] = gene_id  # gene_id as a node label
            graphs_isSick[subjectid]['nodes'][node_pheno] = phenotype  # phenotype as a node label

            # Add edges if they are not already present
            edge_prot = (node_bs, node_prot, protein_score)
            edge_gene = (node_bs, node_gene, gene_score)
            edge_pheno = (node_bs, node_pheno, pheno_score)

            # Use a set to track unique edges
            graphs_isSick[subjectid]['edges'].append(edge_prot)
            graphs_isSick[subjectid]['edges'].append(edge_gene)
            graphs_isSick[subjectid]['edges'].append(edge_pheno)

            # Add edge labels to the edge_labels dictionary
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_prot)] = rel_prot
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_gene)] = rel_gene
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_pheno)] = rel_pheno

    return graphs_isSick

# Connect to your Neo4j database
uri = "neo4j://83.229.84.12"  
user = "tumaiReadonly"
password = "MAKEATHON2024"

driver = connect_to_neo4j(uri, user, password)

# Extract graphs in grakel format
graphs_isSick = extract_graphs_from_neo4j(driver)

graphs_isSick

AttributeError: 'set' object has no attribute 'append'

In [ ]:
## expand the patient graph representations to include Phenotypes and Genes; besides Proteins

from neo4j import GraphDatabase

# Function to connect to Neo4j
def connect_to_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    return driver

# Function to extract graph data from Neo4j
def extract_graphs_from_neo4j(driver):
    query = """
    MATCH (bs:Biological_sample)
    WITH bs LIMIT 10
    MATCH (bs)-[r_prot:HAS_PROTEIN]->(prot:Protein)
    MATCH (bs)-[r_pheno:HAS_PHENOTYPE]->(pheno:Phenotype)
    MATCH (bs)-[r_gene:HAS_DAMAGE]->(gene:Gene)
        OPTIONAL MATCH (bs)-[r_disease:HAS_DISEASE]->(d:Disease)
        //WHERE NOT (bs)-[:HAS_DISEASE]->(d:Disease {name:"control"})
        RETURN DISTINCT bs.subjectid AS subjectid, prot.id AS protein_id, r_prot.score AS protein_score, id(bs) AS node_bs, id(prot) AS node_prot, pheno.id AS phenotype, r_pheno.score AS pheno_score,
        gene.id AS gene, r_gene.score AS gene_score, id(pheno) AS node_pheno, id(gene) AS node_gene, 
        type(r_prot) AS rel_prot, type(r_pheno) AS rel_pheno, type(r_gene) AS rel_gene,
        CASE WHEN d IS NULL THEN 0 ELSE 1 END AS isSick"""
 
    graphs_isSick = {}
    with driver.session(database=NEO4J_DB) as session:
        result = session.run(query)
        
        # Create a dictionary to store nodes and their labels
        for record in result:

            node_bs = record["node_bs"]
            node_prot = record["node_prot"]
            node_gene = record["node_gene"]
            node_pheno = record["node_pheno"]

            rel_prot = record["rel_prot"]
            rel_pheno = record["rel_pheno"]
            rel_gene = record["rel_gene"]

            subjectid = record["subjectid"]
            protein_id = record["protein_id"]
            phenotype = record["phenotype"]
            gene_id = record["gene"]
            
            protein_score = record["protein_score"]
            if protein_score is None:
                protein_score = 1
            else: protein_score = protein_score

            gene_score = record["gene_score"]
            if gene_score is None:
                gene_score = 1
            else: gene_score = gene_score

            pheno_score = record["pheno_score"]
            if pheno_score is None:
                pheno_score = 1 #for now, we don't have a score for phenotypes
            else: pheno_score = pheno_score

            isSick = record["isSick"]
            
            
            
            # If we haven't seen this subjectid before, initialize a new graph entry
            if subjectid not in graphs_isSick:
                graphs_isSick[subjectid] = {
                    'isSick': isSick,
                    'edges': [],
                    'nodes': {},
                    'edge_labels': {}
                }
            
            # Add the phenotype as a node and the relationship (bs -> p) as an edge
            
            #graphs_expanded[subjectid]['isSick'] = isSick
            
            graphs_isSick[subjectid]['nodes'][node_bs] = 'subjectid' #just refers to the Biological_sample node as 'subjectid' so that this node is now considered 'the same' in every patient graph representation
            graphs_isSick[subjectid]['nodes'][node_prot] = protein_id #, protein_score #remove protein_score for now
            graphs_isSick[subjectid]['nodes'][node_gene] = gene_id
            graphs_isSick[subjectid]['nodes'][node_pheno] = phenotype

            graphs_isSick[subjectid]['edges'].append((node_bs, node_prot, protein_score))
            graphs_isSick[subjectid]['edges'].append((node_bs, node_gene, gene_score))
            graphs_isSick[subjectid]['edges'].append((node_bs, node_pheno, pheno_score))

            # Add edge labels to the edge_labels dictionary
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_prot)] = rel_prot
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_gene)] = rel_gene
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_pheno)] = rel_pheno

    return graphs_isSick

# Connect to your Neo4j database
uri = "neo4j://83.229.84.12"  
user = "tumaiReadonly"
password = "MAKEATHON2024"

driver = connect_to_neo4j(uri, user, password)

# Extract graphs in grakel format
graphs_isSick = extract_graphs_from_neo4j(driver)

graphs_isSick


{'10006': {'isSick': 1,
  'edges': [(1687441, 627940),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 422023),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 474243),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 505118),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 548417),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 594227),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 610616),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 421622),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 537183),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 518513),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 468003),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 598760),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 543055),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 595799),
   (1687441, 110815),
   (1687441, 74999),
   (1687441, 620728),
   (1687441, 110815),
   (1687441, 

In [14]:
from grakel import Graph

def transform_for_grakel(graphs_isSick):
    grakel_isSick = []
    labels = []
    
    # Iterate over each graph in your `graphs` dictionary
    for graph in graphs_isSick.values():  
        edges = graph['edges']  # Edges are already in the correct format
        node_labels = graph['nodes']  # Nodes are already in the correct format
        edge_labels = graph['edge_labels']
        
        # Append the graph to the list in the format grakel expects
        gk_graph = Graph(initialization_object=edges, node_labels=node_labels, edge_labels=edge_labels)
        grakel_isSick.append(gk_graph)
        
        # Append the label (isSick) to the labels list
        labels.append(graph['isSick'])
    return grakel_isSick, labels

# Assuming you have the `graphs` dictionary ready as described
# Transform the graphs into the format for grakel
grakel_isSick, a = transform_for_grakel(graphs_isSick)

#grakel_isSick


In [15]:
## Split the data into training and testing sets
from sklearn.model_selection import train_test_split

X_isSick_train, X_isSick_test, y_isSick_train, y_isSick_test =train_test_split(grakel_isSick, a, test_size=0.2, random_state=42)


In [16]:
# shorten code for isSick
import grakel
from grakel import GraphKernel
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import pandas as pd

kernels_isSick = { 'graphlet' : GraphKernel(kernel={"name": "graphlet_sampling", "sampling": {"n_samples": 50}}, random_state=42, normalize=True), 
                    'wl' : GraphKernel(kernel={"name": "weisfeiler_lehman", "n_iter": 1}, normalize=True),
                    'wlop': grakel.WeisfeilerLehmanOptimalAssignment(n_iter=5, normalize=True),
                    'SubgraphMatching': grakel.SubgraphMatching(k=5, normalize=True),
                    'NeighborhoodSubgraphPairwiseDistance': grakel.NeighborhoodSubgraphPairwiseDistance(normalize=True)
                }

classifier_isSick = { 'SVM' : SVC(kernel="precomputed"),
                'RandomForest' : RandomForestClassifier(n_estimators=10, random_state=42)
              }

#node_ids = []
#for graph in X_isSick_test:
    #node_id = graph[0][0][0]
    #node_ids.append(node_id)

with open('task_A_model_results.txt', 'w') as f_report, open('task_A_predictions.txt', 'w') as f_pred:
  for kernel_name, kernel in kernels_isSick.items():
      K_isSick_train = kernel.fit_transform(X_isSick_train)
      K_isSick_test = kernel.transform(X_isSick_test)

      # Train a SVM classifier
      for clf_name, clf in classifier_isSick.items():
          clf.fit(K_isSick_train, y_isSick_train)

          # Predict the labels
          y_isSick_pred = clf.predict(K_isSick_test)
        
          prediction_df = pd.DataFrame({'y_test': y_isSick_test, 'y_pred': y_isSick_pred})
          class_report = classification_report(y_isSick_test, y_isSick_pred)
                     

          # Save the prediction dataframe to the text file in a table-like format
          f_pred.write(f"Kernel: {kernel_name}, Classifier: {clf_name}\n")
          f_pred.write(f"{'y_test':<8} {'y_pred':<8}\n")  # Header row with formatting
          f_pred.write("="*30 + "\n")  # Separator for the table
          
          # Write the data rows in a nicely formatted way
          for _, row in prediction_df.iterrows():
              f_pred.write(f"{row['y_test']:<8} {row['y_pred']:<8}\n")
          f_pred.write("\n" + "="*50 + "\n")  # Divider between different models
          
          # Save the classification report to a text file
          f_report.write(f"Kernel: {kernel_name}, Classifier: {clf_name}\n")
          f_report.write(class_report)
          f_report.write("\n" + "="*50 + "\n")  # Divider for readability

          # Optionally print or save more details as needed
          print(f"Results for Kernel: {kernel_name}, Classifier: {clf_name} saved to files.")




/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_div

Results for Kernel: graphlet, Classifier: SVM saved to files.
Results for Kernel: graphlet, Classifier: RandomForest saved to files.
Results for Kernel: wl, Classifier: SVM saved to files.
Results for Kernel: wl, Classifier: RandomForest saved to files.


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_div

Results for Kernel: wlop, Classifier: SVM saved to files.
Results for Kernel: wlop, Classifier: RandomForest saved to files.


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_div

Results for Kernel: SubgraphMatching, Classifier: SVM saved to files.
Results for Kernel: SubgraphMatching, Classifier: RandomForest saved to files.
Results for Kernel: NeighborhoodSubgraphPairwiseDistance, Classifier: SVM saved to files.
Results for Kernel: NeighborhoodSubgraphPairwiseDistance, Classifier: RandomForest saved to files.


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_div

In [17]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, classification_report, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from kernel_wrappers import WeisfeilerLehmanWrapperA, GraphletSamplingWrapperA, SubgraphMatchingWrapperA, WeisfeilerLehmanOAWrapperA, NeighborhoodSubgraphPairwiseDistanceWrapperA
import grakel
from grakel import GraphKernel, WeisfeilerLehman, GraphletSampling, SubgraphMatching, WeisfeilerLehmanOptimalAssignment, NeighborhoodSubgraphPairwiseDistance

from neo4j import GraphDatabase
from pandas import DataFrame
import csv
from neo4j.debug import watch
import os
import warnings
from sklearn.exceptions import ConvergenceWarning
# Ignore ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)


# Step 1: Kernel Tuning
kernels_A = {
    'WeisfeilerLehman': WeisfeilerLehmanWrapperA(),
    'GraphletSampling': GraphletSamplingWrapperA()
    #'SubgraphMatching': SubgraphMatchingWrapperA(),
    #'WeisfeilerLehmanOptimalAssignment': WeisfeilerLehmanOAWrapperA(),
    #'NeighborhoodSubgraphPairwiseDistance': NeighborhoodSubgraphPairwiseDistanceWrapperA()
}

kernel_param_grids = {
    'WeisfeilerLehman': {'n_iter': [1, 3, 5]},
    'GraphletSampling': {'n_samples': [50, 100, 200, 500]},
    'SubgraphMatching': {'k': [3, 5, 7, 10]},
    'WeisfeilerLehmanOptimalAssignment': {'n_iter': [1, 3, 5]},
    'NeighborhoodSubgraphPairwiseDistance': {'r': [3, 5, 7], 'd': [3, 4, 5, 7]}
}

best_kernels_A = {}
for kernel_name, kernel in kernels_A.items():
    print(f"Tuning kernel: {kernel_name}")
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search_kernel = GridSearchCV(estimator=kernel, param_grid=kernel_param_grids[kernel_name], cv=cv)
    grid_search_kernel.fit(X_isSick_train, y_isSick_train)
    best_kernels_A[kernel_name] = {
        'best_estimator': grid_search_kernel.best_estimator_,
        'best_params': grid_search_kernel.best_params_,
    }
    print(f"Best {kernel_name} Params: {grid_search_kernel.best_params_}")

# Step 2: Classifier Tuning
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),	
    'KNeighborsClassifier': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(random_state=42)
    #'SVC': SVC(kernel="precomputed")
}

classifier_param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [10, 50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [10, 50, 100, 200],
        'learning_rate': [0.1, 0.01, 0.001],
        'max_depth': [3, 10, 20]
    },
    'DecisionTreeClassifier': {
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10]
    },
    'KNeighborsClassifier': {
        'n_neighbors': [3, 5, 7, 10],
        'weights': ['uniform', 'distance'],
        'p': [1, 2]
    },
    'LogisticRegression': {
        'C': [0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
    },
        'SVC': {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto']
    }
}

results_A = {}
for kernel_name, kernel_info in best_kernels_A.items():
    best_kernel = kernel_info['best_estimator']
    K_isSick_train = best_kernel.fit_transform(X_isSick_train)
    K_isSick_test = best_kernel.transform(X_isSick_test)
    
    for clf_name, clf in classifiers.items():
        print(f"Tuning classifier: {clf_name} with kernel: {kernel_name}")
        grid_search_clf = GridSearchCV(estimator=clf, param_grid=classifier_param_grids[clf_name], cv=3)
        grid_search_clf.fit(K_isSick_train, y_isSick_train)
        best_clf = grid_search_clf.best_estimator_
        
        # Evaluate on the test set
        y_isSick_pred = best_clf.predict(K_isSick_test)
        accuracy = accuracy_score(y_isSick_test, y_isSick_pred)
        
        # Store results
        results_A[f"{kernel_name} + {clf_name}"] = {
            'kernel_params': kernel_info['best_params'],
            'classifier_params': grid_search_clf.best_params_,
            'accuracy': accuracy
        }
        print(f"Accuracy for {kernel_name} + {clf_name}: {accuracy}")

# Display all results
for combination, result in results_A.items():
    print(f"Combination: {combination}")
    print(f"Kernel Params: {result['kernel_params']}")
    print(f"Classifier Params: {result['classifier_params']}")
    print(f"Accuracy: {result['accuracy']}\n")


Tuning kernel: WeisfeilerLehman


ValueError: 
All the 9 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/kernel_wrappers.py", line 16, in fit
    self.K_isSick_train_ = self.kernel.fit_transform(X)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/utils/_set_output.py", line 316, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
              

In [12]:
# Initialize variables to track the best combination
best_combination = None
best_accuracy = 0
best_y_pred = None
best_classification_report = None
best_confusion_matrix = None

# Iterate through results to find the best combination
for combination, result in results_A.items():
    if result['accuracy'] > best_accuracy:
        best_accuracy = result['accuracy']
        best_combination = combination
        
        # Extract kernel and classifier from the best combination
        kernel_name, clf_name = combination.split(" + ")
        best_kernel = best_kernels_A[kernel_name]['best_estimator']
        best_clf = classifiers[clf_name]
        
        # Fit the best kernel and classifier
        K_isSick_train = best_kernel.fit_transform(X_isSick_train)
        K_isSick_test = best_kernel.transform(X_isSick_test)
        best_clf.set_params(**result['classifier_params'])  # Apply best parameters
        best_clf.fit(K_isSick_train, y_isSick_train)
        
        # Generate predictions
        best_y_pred = best_clf.predict(K_isSick_test)
        
        # Generate reports
        best_classification_report = classification_report(y_isSick_test, best_y_pred)
        best_confusion_matrix = confusion_matrix(y_isSick_test, best_y_pred)

# Output the best combination and its details
print(f"Best Combination: {best_combination}")
print(f"Best Accuracy: {best_accuracy}")
print("\nClassification Report for Best Combination:")
print(best_classification_report)
print("\nConfusion Matrix for Best Combination:")
print(best_confusion_matrix)

# Return the predictions compared to y_test
print("\nPredictions vs. Ground Truth:")
comparison_df = pd.DataFrame({'y_test': y_isSick_test, 'y_pred': best_y_pred})
print(comparison_df)


Best Combination: WeisfeilerLehman + RandomForestClassifier
Best Accuracy: 0.8

Classification Report for Best Combination:
              precision    recall  f1-score   support

           0       0.67      0.67      0.67         6
           1       0.86      0.86      0.86        14

    accuracy                           0.80        20
   macro avg       0.76      0.76      0.76        20
weighted avg       0.80      0.80      0.80        20


Confusion Matrix for Best Combination:
[[ 4  2]
 [ 2 12]]

Predictions vs. Ground Truth:
    y_test  y_pred
0        0       0
1        0       0
2        1       1
3        1       1
4        1       1
5        1       1
6        0       1
7        1       1
8        1       0
9        1       1
10       0       0
11       1       1
12       0       0
13       0       1
14       1       1
15       1       1
16       1       0
17       1       1
18       1       1
19       1       1


try to predict ICD10 letter

In [ ]:
# new patient representations

from neo4j import GraphDatabase

# Function to connect to Neo4j
def connect_to_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    return driver

# Function to extract ICD10 patient graphs from Neo4j
def extract_icd10_graphs(driver):
    icd10_query = """
    MATCH (bs:Biological_sample)-[r_prot:HAS_PROTEIN]->(prot:Protein)
    MATCH (bs)-[r_pheno:HAS_PHENOTYPE]->(pheno:Phenotype)
    MATCH (bs)-[r_gene:HAS_DAMAGE]->(gene:Gene)
    OPTIONAL MATCH (bs)-[:HAS_DISEASE]->(d:Disease)
    WITH bs, r_prot, prot, r_pheno, pheno, r_gene, gene, d, CASE WHEN d IS NOT NULL THEN [s in d.synonyms WHERE s STARTS WITH "ICD10CM" | s] ELSE ["CTL"] END as ICD10
    RETURN DISTINCT bs.subjectid AS subjectid, prot.id AS protein_id, r_prot.score AS protein_score, id(bs) AS node_bs, id(prot) AS node_prot, pheno.id AS phenotype, r_pheno.score AS pheno_score, 
    gene.id AS gene, r_gene.score AS gene_score, id(pheno) AS node_pheno, id(gene) AS node_gene,
    type(r_prot) AS rel_prot, type(r_pheno) AS rel_pheno, type(r_gene) AS rel_gene,
    COLLECT(ICD10[0]) as icd10
    """
    # alternative query: https://github.com/LaurenzSommerlad/TUM.ai-Makeathon2024-Amigo-Challenge/blob/main/dataprocess.py

    graphs_icd10 = {}
    with driver.session(database=NEO4J_DB) as session:
        result_icd10 = session.run(icd10_query)

        # Create a dictionary to store nodes and their labels
        for record in result_icd10:

            node_bs = record["node_bs"]
            node_prot = record["node_prot"]
            node_gene = record["node_gene"]
            node_pheno = record["node_pheno"]

            subjectid = record["subjectid"]
            protein_id = record["protein_id"]
            phenotype = record["phenotype"]
            gene_id = record["gene"]

            rel_prot = record["rel_prot"]
            rel_gene = record["rel_gene"]
            rel_pheno = record["rel_pheno"]
            
            protein_score = record["protein_score"]
            if protein_score is None:
                protein_score = 1
            else: protein_score = protein_score

            gene_score = record["gene_score"]
            if gene_score is None:
                gene_score = 1
            else: gene_score = gene_score
            
            pheno_score = record["pheno_score"]
            if pheno_score is None:
                pheno_score = 1 #for now, we don't have a score for phenotypes
            else: pheno_score = pheno_score

            icd10 = record["icd10"]

            if isinstance(icd10, list) and icd10:
                icd10 = icd10[0]
                icd10 = icd10.replace("ICD10CM:", "")
                icd10 = icd10[0]
            elif icd10 == []:
                icd10 = "NaN"
            else:
                icd10 = "CTL"
                
            
            # If we haven't seen this subjectid before, initialize a new graph entry
            if subjectid not in graphs_icd10:
                graphs_icd10[subjectid] = {
                    'icd10': icd10,
                    'edges': [],
                    'edge_labels': {},
                    'nodes': {}
                }
            
            # Add the phenotype as a node and the relationship (bs -> p) as an edge
            #graphs_icd10[subjectid]['icd10'] = icd10

            graphs_icd10[subjectid]['nodes'][node_bs] = 'subjectid'
            graphs_icd10[subjectid]['nodes'][node_prot] = protein_id
            graphs_icd10[subjectid]['nodes'][node_gene] = gene_id
            graphs_icd10[subjectid]['nodes'][node_pheno] = phenotype


            graphs_icd10[subjectid]['edges'].append((node_bs, node_prot))
            graphs_icd10[subjectid]['edges'].append((node_bs, node_gene))
            graphs_icd10[subjectid]['edges'].append((node_bs, node_pheno))

            # Add edge labels to the edge_labels dictionary
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_prot)] = rel_prot
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_gene)] = rel_gene
            graphs_isSick[subjectid]['edge_labels'][(node_bs, node_pheno)] = rel_pheno
            
    return graphs_icd10

# Connect to your Neo4j database
uri = "neo4j://83.229.84.12"  
user = "tumaiReadonly"
password = "MAKEATHON2024"

driver = connect_to_neo4j(uri, user, password)

# Extract graphs in grakel format
graphs_icd10 = extract_icd10_graphs(driver)

# Print the graphs
#for subjectid, graph in graphs_icd10.items():
#    print(f"Biological Sample: {subjectid}")
#    print("ICD10:", graph['icd10'])
#    print("Nodes (Proteins):", graph['nodes'])
#    print("Edges (bs -> Protein):", graph['edges'])
#    print("Edge Features (Protein Score):", graph['edge_features'])
#    print()

graphs_icd10



{'10006': {'icd10': 'D',
  'edges': [(1687441, 627940, 16.994526408399366),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 422023, 13.74736819523299),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 474243, 19.959805818499774),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 505118, 11.910361000968935),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 548417, 6.599672569330909),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 594227, 3.7231582809872554),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 610616, 6.7557588233353805),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 421622, 16.034270459270566),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 537183, 7.80656761783275),
   (1687441, 110815, 6.91694148560063),
   (1687441, 74999, 1),
   (1687441, 518

In [14]:
# drop all patients where icd10 is NaN

graphs_icd10 = {key: value for key, value in graphs_icd10.items() if value['icd10'] != "NaN"}

#graphs_icd10

In [15]:
len(graphs_icd10)

96

In [16]:
from sklearn.preprocessing import LabelEncoder
from grakel import Graph

def transform_for_grakel(graphs_icd10):
    grakel_icd10 = []
    labels = []
    
    # Iterate over each graph in your `graphs` dictionary
    for graph in graphs_icd10.values():  
        edges = graph['edges']  # Edges are already in the correct format
        node_labels = graph['nodes']  # Nodes are already in the correct format
        
        # Append the graph to the list in the format grakel expects
        grakel_graph = Graph(initialization_object=edges, node_labels=node_labels)
        grakel_icd10.append(grakel_graph)
        
        # Append the label (isSick) to the labels list
        labels.append(graph['icd10'])
    return grakel_icd10, labels

# Assuming you have the `graphs` dictionary ready as described
# Transform the graphs into the format for grakel
grakel_icd10, b = transform_for_grakel(graphs_icd10)

label_encoder = LabelEncoder()
b = label_encoder.fit_transform(b)

#grakel_icd10


In [17]:
len(b)

96

In [18]:
from grakel import GraphKernel
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import pandas as pd

# Split the data into training and testing sets
X_icd10_train, X_icd10_test, y_icd10_train, y_icd10_test = train_test_split(grakel_icd10, b, test_size=0.2, random_state=42)

print(len(y_icd10_test))


20


In [27]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, classification_report, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from kernel_wrappers import WeisfeilerLehmanWrapperB, GraphletSamplingWrapperB, SubgraphMatchingWrapperB, WeisfeilerLehmanOAWrapperB, NeighborhoodSubgraphPairwiseDistanceWrapperB
import grakel
from grakel import GraphKernel, WeisfeilerLehman, GraphletSampling, SubgraphMatching, WeisfeilerLehmanOptimalAssignment, NeighborhoodSubgraphPairwiseDistance

from neo4j import GraphDatabase
from pandas import DataFrame
import csv
from neo4j.debug import watch
import os
import warnings
from sklearn.exceptions import ConvergenceWarning
# Ignore ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)



# Step 1: Kernel Tuning
kernels_B = {
    'WeisfeilerLehman': WeisfeilerLehmanWrapperB(),
    'GraphletSampling': GraphletSamplingWrapperB()
    #'SubgraphMatching': SubgraphMatchingWrapperB()
    #'WeisfeilerLehmanOptimalAssignment': WeisfeilerLehmanOAWrapperB(),
    #'NeighborhoodSubgraphPairwiseDistance': NeighborhoodSubgraphPairwiseDistanceWrapperB()
}

kernel_param_grids = {
    'WeisfeilerLehman': {'n_iter': [1, 3, 5]},
    'GraphletSampling': {'n_samples': [50, 100, 200, 500]},
    'SubgraphMatching': {'k': [3, 5, 7, 10]},
    'WeisfeilerLehmanOptimalAssignment': {'n_iter': [1, 3, 5]},
    'NeighborhoodSubgraphPairwiseDistance': {'r': [3, 5, 7], 'd': [3, 4, 5, 7]}
}

best_kernels_B = {}
for kernel_name, kernel in kernels_B.items():
    print(f"Tuning kernel: {kernel_name}")
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search_kernel = GridSearchCV(estimator=kernel, param_grid=kernel_param_grids[kernel_name], cv=cv)
    grid_search_kernel.fit(X_icd10_train, y_icd10_train)
    best_kernels_B[kernel_name] = {
        'best_estimator': grid_search_kernel.best_estimator_,
        'best_params': grid_search_kernel.best_params_,
    }
    print(f"Best {kernel_name} Params: {grid_search_kernel.best_params_}")

# Step 2: Classifier Tuning
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),	
    'KNeighborsClassifier': KNeighborsClassifier(),
    #'LogisticRegression': LogisticRegression(random_state=42)
    #'SVC': SVC(kernel="precomputed")
}

classifier_param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [10, 50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [10, 50, 100, 200],
        'learning_rate': [0.1, 0.01, 0.001],
        'max_depth': [3, 10, 20]
    },
    'DecisionTreeClassifier': {
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10]
    },
    'KNeighborsClassifier': {
        'n_neighbors': [3, 5, 7, 10],
        'weights': ['uniform', 'distance'],
        'p': [1, 2]
    },
    'LogisticRegression': {
        'C': [0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
    },
        'SVC': {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto']
    }
}

results_B = {}
for kernel_name, kernel_info in best_kernels_B.items():
    best_kernel = kernel_info['best_estimator']
    K_icd10_train = best_kernel.fit_transform(X_icd10_train)
    K_icd10_test = best_kernel.transform(X_icd10_test)
    
    for clf_name, clf in classifiers.items():
        print(f"Tuning classifier: {clf_name} with kernel: {kernel_name}")
        grid_search_clf = GridSearchCV(estimator=clf, param_grid=classifier_param_grids[clf_name], cv=3)
        grid_search_clf.fit(K_icd10_train, y_icd10_train)
        best_clf = grid_search_clf.best_estimator_
        
        # Evaluate on the test set
        y_icd10_pred = best_clf.predict(K_icd10_test)
        #accuracy = accuracy_score(y_test, y_pred)
        accuracy = accuracy_score(y_icd10_test, y_icd10_pred)
        
        # Store results
        results_B[f"{kernel_name} + {clf_name}"] = {
            'kernel_params': kernel_info['best_params'],
            'classifier_params': grid_search_clf.best_params_,
            'accuracy': accuracy
        }
        print(f"Accuracy for {kernel_name} + {clf_name}: {accuracy}")


# Display all results
for combination, result in results_B.items():
    print(f"Combination: {combination}")
    print(f"Kernel Params: {result['kernel_params']}")
    print(f"Classifier Params: {result['classifier_params']}")
    print(f"Accuracy: {result['accuracy']}\n")


Tuning kernel: WeisfeilerLehman


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Best WeisfeilerLehman Params: {'n_iter': 3}
Tuning kernel: GraphletSampling


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Best GraphletSampling Params: {'n_samples': 100}
Tuning classifier: RandomForestClassifier with kernel: WeisfeilerLehman


...

Accuracy for GraphletSampling + RandomForestClassifier: 0.25
Tuning classifier: GradientBoostingClassifier with kernel: GraphletSampling


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Accuracy for GraphletSampling + GradientBoostingClassifier: 0.3
Tuning classifier: DecisionTreeClassifier with kernel: GraphletSampling
Accuracy for GraphletSampling + DecisionTreeClassifier: 0.2
Tuning classifier: KNeighborsClassifier with kernel: GraphletSampling


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Accuracy for GraphletSampling + KNeighborsClassifier: 0.2
Combination: WeisfeilerLehman + RandomForestClassifier
Kernel Params: {'n_iter': 3}
Classifier Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 10}
Accuracy: 0.3

Combination: WeisfeilerLehman + GradientBoostingClassifier
Kernel Params: {'n_iter': 3}
Classifier Params: {'learning_rate': 0.001, 'max_depth': 3, 'n_estimators': 10}
Accuracy: 0.35

Combination: WeisfeilerLehman + DecisionTreeClassifier
Kernel Params: {'n_iter': 3}
Classifier Params: {'max_depth': None, 'min_samples_split': 5}
Accuracy: 0.45

Combination: WeisfeilerLehman + KNeighborsClassifier
Kernel Params: {'n_iter': 3}
Classifier Params: {'n_neighbors': 10, 'p': 1, 'weights': 'distance'}
Accuracy: 0.25

Combination: GraphletSampling + RandomForestClassifier
Kernel Params: {'n_samples': 100}
Classifier Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 10}
Accuracy: 0.25

Combination: GraphletSampling + GradientBoostingClassifie

In [28]:
# Initialize variables to track the best combination
best_combination = None
best_accuracy = 0
best_y_pred = None
best_classification_report = None
best_confusion_matrix = None

# Iterate through results to find the best combination
for combination, result in results_B.items():
    if result['accuracy'] > best_accuracy:
        best_accuracy = result['accuracy']
        best_combination = combination
        
        # Extract kernel and classifier from the best combination
        kernel_name, clf_name = combination.split(" + ")
        best_kernel = best_kernels_B[kernel_name]['best_estimator']
        best_clf = classifiers[clf_name]
        
        # Fit the best kernel and classifier
        K_icd10_train = best_kernel.fit_transform(X_icd10_train)
        K_icd10_test = best_kernel.transform(X_icd10_test)
        best_clf.set_params(**result['classifier_params'])  # Apply best parameters
        best_clf.fit(K_icd10_train, y_icd10_train)
        
        # Generate predictions
        best_y_pred = best_clf.predict(K_icd10_test)

        y_letter = label_encoder.inverse_transform(y_icd10_test)
        y_pred_letter = label_encoder.inverse_transform(best_y_pred)
        
        # Generate reports
        best_classification_report = classification_report(y_letter, y_pred_letter)
        best_confusion_matrix = confusion_matrix(y_icd10_test, best_y_pred)

# Output the best combination and its details
print(f"Best Combination: {best_combination}")
print(f"Best Accuracy: {best_accuracy}")
print("\nClassification Report for Best Combination:")
print(best_classification_report)
print("\nConfusion Matrix for Best Combination:")
print(best_confusion_matrix)

# Return the predictions compared to y_test
print("\nPredictions vs. Ground Truth:")
comparison_df = pd.DataFrame({'y_letter':y_letter, 'pred_letter': y_pred_letter, 'y_test': y_icd10_test, 'y_pred': best_y_pred})
print(comparison_df)


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_div

Best Combination: WeisfeilerLehman + DecisionTreeClassifier
Best Accuracy: 0.45

Classification Report for Best Combination:
              precision    recall  f1-score   support

           C       0.62      0.71      0.67         7
           D       0.00      0.00      0.00         2
           E       0.00      0.00      0.00         1
           I       0.38      0.60      0.46         5
           J       0.00      0.00      0.00         2
           K       0.50      0.50      0.50         2
           R       0.00      0.00      0.00         1

    accuracy                           0.45        20
   macro avg       0.21      0.26      0.23        20
weighted avg       0.36      0.45      0.40        20


Confusion Matrix for Best Combination:
[[5 0 0 1 0 1 0]
 [0 0 0 0 2 0 0]
 [0 0 0 1 0 0 0]
 [2 0 0 3 0 0 0]
 [1 0 0 1 0 0 0]
 [0 0 0 1 0 1 0]
 [0 0 0 1 0 0 0]]

Predictions vs. Ground Truth:
   y_letter pred_letter  y_test  y_pred
0         C           C       0       0
1      

In [22]:

kernels = { 'graphlet' : GraphKernel(kernel={"name": "graphlet_sampling", "sampling": {"n_samples": 250}}, random_state=42, normalize=True), 
            'wl' : GraphKernel(kernel={"name": "weisfeiler_lehman", "n_iter": 5}, normalize=True)
          }

classifier = { 'SVM' : SVC(kernel="precomputed"),
                'RandomForest' : RandomForestClassifier(n_estimators=100, random_state=42)
              }

#node_ids = []
#for graph in X_icd10_test:
#    node_id = graph[0][0][0]
#    node_ids.append(node_id)

with open('task_B_model_results.txt', 'w') as f_report, open('task_B_predictions.txt', 'w') as f_pred:
  for kernel_name, kernel in kernels.items():
      K_icd10_train = kernel.fit_transform(X_icd10_train)
      K_icd10_test = kernel.transform(X_icd10_test)

      # Train a SVM classifier
      for clf_name, clf in classifier.items():
          clf.fit(K_icd10_train, y_icd10_train)

          # Predict the labels
          y_icd10_pred = clf.predict(K_icd10_test)

          # Inverse transform predictions and actual values back to original labels
          y_test_letter = label_encoder.inverse_transform(y_icd10_test)
          y_pred_letter = label_encoder.inverse_transform(y_icd10_pred)
        
          prediction_df = pd.DataFrame({'letter': y_test_letter, 'letter_pred': y_pred_letter, 'y_test': y_icd10_test, 'y_pred': y_icd10_pred})
          class_report = classification_report(y_test_letter, y_pred_letter)
                     

          # Save the prediction dataframe to the text file in a table-like format
          f_pred.write(f"Kernel: {kernel_name}, Classifier: {clf_name}\n")
          f_pred.write(f"{'Letter':<8} {'Predicted Letter':<8} {'y_test':<8} {'y_pred':<8}\n")  # Header row with formatting
          f_pred.write("="*30 + "\n")  # Separator for the table
          
          # Write the data rows in a nicely formatted way
          for _, row in prediction_df.iterrows():
              f_pred.write(f"{row['letter']:<8} {row['letter_pred']:<8} {row['y_test']:<8} {row['y_pred']:<8}\n")
          f_pred.write("\n" + "="*50 + "\n")  # Divider between different models
          
          # Save the classification report to a text file
          f_report.write(f"Kernel: {kernel_name}, Classifier: {clf_name}\n")
          f_report.write(class_report)
          f_report.write("\n" + "="*50 + "\n")  # Divider for readability

          # Optionally print or save more details as needed
          print(f"Results for Kernel: {kernel_name}, Classifier: {clf_name} saved to files.")




/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_div

Results for Kernel: graphlet, Classifier: SVM saved to files.
Results for Kernel: graphlet, Classifier: RandomForest saved to files.
Results for Kernel: wl, Classifier: SVM saved to files.
Results for Kernel: wl, Classifier: RandomForest saved to files.


/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ssc/projects/grakel/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_div

In [24]:
print(len(y_icd10_test))

20
